# Time Series Analysis

Topics:
- `pd.to_datetime()` — parsing dates
- Setting datetime as index
- `.dt` accessor — extracting date parts
- Resampling — grouping by time period
- Rolling windows — moving averages
- Shifting — lag/lead comparisons

In [1]:
import pandas as pd
import numpy as np

## 1. Parsing Dates — `pd.to_datetime()`

Pandas stores dates as `datetime64` — a special dtype that unlocks time-based operations.
Without this, dates are just strings and you can't do any time math.

In [2]:
# String dates — pandas doesn't know these are dates yet
dates = ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05"]

s = pd.Series(dates)
print("Before:", s.dtype)  # object (string)

s = pd.to_datetime(s)
print("After: ", s.dtype)  # datetime64

Before: str
After:  datetime64[us]


In [3]:
# pandas can parse many formats automatically
# but the dates should be in same format in list or we should use format="mixed" parameter
print(pd.to_datetime(["01/15/2024", "Jan 15 2024", "2024-01-15", "15-01-2024"], format="mixed"))

# never mix date formats in real data. If you receive DD-MM-YYYY format, specify it explicitly:
pd.to_datetime(["15-01-2024", "16-01-2024"], format="%d-%m-%Y")


DatetimeIndex(['2024-01-15', '2024-01-15', '2024-01-15', '2024-01-15'], dtype='datetime64[us]', freq=None)


DatetimeIndex(['2024-01-15', '2024-01-16'], dtype='datetime64[us]', freq=None)

In [4]:
# if the format is unusual, specify it explicitly
pd.to_datetime(["15/01/2024"], format="%d/%m/%Y")

DatetimeIndex(['2024-01-15'], dtype='datetime64[us]', freq=None)

## 2. Building a Time Series Dataset

```python
pd.date_range()


dates = pd.date_range(start="2024-01-01", periods=365, freq="D")
start — where the sequence begins
periods — how many dates to generate (365 = full year)
freq="D" — interval between each date (D = daily)
Result: [2024-01-01, 2024-01-02, 2024-01-03 ... 2024-12-31]

np.random.seed(42)

Sets a fixed seed so random numbers are reproducible — anyone running this code gets the same "random" data. 42 is just a convention, any number works.

np.random.randint(1000, 5000, size=365)

Generates 365 random integers between 1000 and 5000.

+ np.arange(365) * 5

np.arange(365) = [0, 1, 2, 3 ... 364]

Multiplied by 5 = [0, 5, 10, 15 ... 1820]

Added to revenue — each day gets slightly higher baseline. This is what creates the upward trend in the data. Without this, revenue would just be flat random noise.


day 1:   random(1000-5000) + 0
day 2:   random(1000-5000) + 5
day 3:   random(1000-5000) + 10
...
day 365: random(1000-5000) + 1820
np.random.choice(["North", "South", "East"], size=365)

Randomly picks one of the 3 regions for each of the 365 rows.

In [5]:
# pd.date_range — generate a sequence of dates
dates = pd.date_range(start="2024-01-01", periods=365, freq="D")

np.random.seed(42)
df = pd.DataFrame({
    "date":    dates,
    "revenue": np.random.randint(1000, 5000, size=365) + np.arange(365) * 5,  # upward trend
    "users":   np.random.randint(100, 500, size=365),
    "region":  np.random.choice(["North", "South", "East"], size=365)
})

df.head()

,date,revenue,users,region
0,2024-01-01,4174,159,North
1,2024-01-02,4512,468,East
2,2024-01-03,1870,101,North
3,2024-01-04,2309,484,North
4,2024-01-05,2150,403,South


## 3. Setting Datetime as Index

Setting the date column as the index unlocks time-based slicing and resampling.
This is the standard way to work with time series in pandas.

In [6]:
# syntax: dataframe.set_index("col_name")
df = df.set_index("date")
df.head()

,revenue,users,region
date,,,
2024-01-01,4174,159,North
2024-01-02,4512,468,East
2024-01-03,1870,101,North
2024-01-04,2309,484,North
2024-01-05,2150,403,South


In [11]:
# time-based slicing — only works when date is the index
df.loc["2024-03"]                        # all of March


,revenue,users,region
date,,,
2024-03-01,5190,150,North
2024-03-02,1951,363,South
2024-03-03,3378,382,South
2024-03-04,4203,126,South
2024-03-05,3534,325,North
2024-03-06,2622,376,East
2024-03-07,3765,385,North
2024-03-08,1935,196,East
2024-03-09,3703,383,North


In [12]:
df.loc["2024-01-01" : "2024-01-31"]      # January only

,revenue,users,region
date,,,
2024-01-01,4174,159,North
2024-01-02,4512,468,East
2024-01-03,1870,101,North
2024-01-04,2309,484,North
2024-01-05,2150,403,South
2024-01-06,2120,353,South
2024-01-07,4802,239,East
2024-01-08,4127,136,South
2024-01-09,2678,259,East


## 4. The `.dt` Accessor — Extracting Date Parts

`.dt` gives you access to date components (month, day, weekday, etc.) from a datetime column.
Think of it like `.str` but for dates.

In [13]:
# reset index to use .dt on a column (not index)
df_col = df.reset_index()

df_col["month"]     = df_col["date"].dt.month        # 1-12
df_col["month_name"]= df_col["date"].dt.month_name() # January, February...
df_col["day"]       = df_col["date"].dt.day           # 1-31
df_col["weekday"]   = df_col["date"].dt.day_name()    # Monday, Tuesday...
df_col["week"]      = df_col["date"].dt.isocalendar().week  # week number
df_col["quarter"]   = df_col["date"].dt.quarter       # 1-4
df_col["is_weekend"]= df_col["date"].dt.dayofweek >= 5  # True/False

df_col[["date", "month_name", "weekday", "quarter", "is_weekend"]].head(10)

,date,month_name,weekday,quarter,is_weekend
0,2024-01-01,January,Monday,1,False
1,2024-01-02,January,Tuesday,1,False
2,2024-01-03,January,Wednesday,1,False
3,2024-01-04,January,Thursday,1,False
4,2024-01-05,January,Friday,1,False
5,2024-01-06,January,Saturday,1,True
6,2024-01-07,January,Sunday,1,True
7,2024-01-08,January,Monday,1,False
8,2024-01-09,January,Tuesday,1,False
9,2024-01-10,January,Wednesday,1,False


In [14]:
# practical use: which weekday has highest average revenue?
df_col.groupby("weekday")["revenue"].mean().sort_values(ascending=False)

weekday
Sunday       4169.423077
Monday       4132.849057
Friday       4026.057692
Tuesday      4024.923077
Wednesday    3867.038462
Saturday     3863.557692
Thursday     3809.807692
Name: revenue, dtype: float64

## 5. Resampling — Grouping by Time Period

`resample()` is like `groupby()` but for time periods.
It collapses rows into time buckets (weekly, monthly, quarterly, etc.).

Common freq aliases:
- `"D"` — daily
- `"W"` — weekly
- `"ME"` — month end
- `"QE"` — quarter end
- `"YE"` — year end

```python
When you resample, pandas needs to pick one date to represent each bucket. "End" means it uses the last day of that period as the label.

df["revenue"].resample("ME").sum()

# date label used is the LAST day of each month:
# 2024-01-31    total revenue for all of January
# 2024-02-29    total revenue for all of February
# 2024-03-31    total revenue for all of March

Same idea for quarter end:


df["revenue"].resample("QE").sum()

# 2024-03-31    Q1 total (Jan + Feb + Mar)
# 2024-06-30    Q2 total (Apr + May + Jun)
# 2024-09-30    Q3 total (Jul + Aug + Sep)
# 2024-12-31    Q4 total (Oct + Nov + Dec)
Why "end" and not "start"? There's also "MS" (month start), "QS" (quarter start) — those label each bucket with the first day instead:


"MS"  →  2024-01-01, 2024-02-01, 2024-03-01 ...
"ME"  →  2024-01-31, 2024-02-29, 2024-03-31 ...
Same data grouped the same way — only the label date differs.

In [15]:
# monthly total revenue
df["revenue"].resample("ME").sum()

date
2024-01-31     97111
2024-02-29     96610
2024-03-31    109751
2024-04-30    114187
2024-05-31    108033
2024-06-30    121180
2024-07-31    121117
2024-08-31    136455
2024-09-30    130198
2024-10-31    141591
2024-11-30    138984
2024-12-31    139386
Freq: ME, Name: revenue, dtype: int64

In [16]:
# weekly average users
df["users"].resample("W").mean()

date
2024-01-07    315.285714
2024-01-14    240.285714
2024-01-21    233.571429
2024-01-28    274.000000
2024-02-04    368.000000
2024-02-11    294.428571
2024-02-18    396.857143
2024-02-25    290.857143
2024-03-03    297.428571
2024-03-10    322.428571
2024-03-17    248.714286
2024-03-24    240.714286
2024-03-31    339.000000
2024-04-07    259.428571
2024-04-14    297.571429
2024-04-21    293.714286
2024-04-28    345.000000
2024-05-05    302.857143
2024-05-12    335.285714
2024-05-19    317.142857
2024-05-26    343.571429
2024-06-02    270.428571
2024-06-09    309.571429
2024-06-16    276.000000
2024-06-23    256.000000
2024-06-30    368.571429
2024-07-07    287.428571
2024-07-14    304.571429
2024-07-21    350.285714
2024-07-28    232.142857
2024-08-04    316.428571
2024-08-11    299.000000
2024-08-18    262.285714
2024-08-25    319.285714
2024-09-01    282.428571
2024-09-08    299.142857
2024-09-15    340.714286
2024-09-22    316.857143
2024-09-29    351.714286
2024-10-06    261.42

In [17]:
# multiple aggregations at once
df["revenue"].resample("ME").agg(["sum", "mean", "max", "min"])

,sum,mean,max,min
date,,,,
2024-01-31,97111,3132.612903,4825,1225
2024-02-29,96610,3331.379310,4963,1191
2024-03-31,109751,3540.354839,5388,1454
2024-04-30,114187,3806.233333,5474,1676
2024-05-31,108033,3484.935484,5618,1739
2024-06-30,121180,4039.333333,5745,2000
2024-07-31,121117,3907.000000,5742,2110
2024-08-31,136455,4401.774194,6029,2930
2024-09-30,130198,4339.933333,6218,2597


In [18]:
# quarterly revenue
df["revenue"].resample("QE").sum()

date
2024-03-31    303472
2024-06-30    343400
2024-09-30    387770
2024-12-31    419961
Freq: QE-DEC, Name: revenue, dtype: int64

## 6. Rolling Windows — Moving Averages


`rolling(n)` creates a sliding window of n rows that moves through the data one step at a time.
At each row, it looks back n rows and computes the aggregation over that window.
Used to smooth out noise and reveal the underlying trend.

---

### How the window moves

```
revenue = [100, 200, 150, 300, 250]

rolling(3).mean():
row 0: NaN          (only 1 row available, window not full)
row 1: NaN          (only 2 rows available, window not full)
row 2: (100+200+150) / 3 = 150.0   ← first full window
row 3: (200+150+300) / 3 = 216.7   ← window slides forward
row 4: (150+300+250) / 3 = 233.3   ← window slides again
```

The first `n-1` rows are always `NaN` because there aren't enough prior rows to fill the window.

---

### Why use it?

Raw daily data is noisy — one bad day or one spike makes it hard to see the real trend.
A rolling average smooths that out.

- **7-day rolling avg** — removes day-of-week effects (weekends vs weekdays)
- **30-day rolling avg** — shows the monthly trend
- The longer the window, the smoother but more lagged the result

---

### Key parameters

| Parameter | What it does | Default |
|-----------|-------------|---------|
| `window` | number of rows in the window | required |
| `min_periods` | minimum rows needed to compute (avoids leading NaNs) | same as window |
| `center` | if True, window is centered around current row instead of trailing | `False` |

```python
# min_periods — compute even if window isn't full yet
df["revenue"].rolling(7, min_periods=1).mean()   # no leading NaNs

# center — window looks both forward and backward
df["revenue"].rolling(7, center=True).mean()
```

---

Real-world use: stock prices, sales trends, server metrics.

In [19]:
# 7-day rolling average revenue
df["revenue_7d"]  = df["revenue"].rolling(7).mean()

# 30-day rolling average
df["revenue_30d"] = df["revenue"].rolling(30).mean()

df[["revenue", "revenue_7d", "revenue_30d"]].head(35)
df

,revenue,users,region,revenue_7d,revenue_30d
date,,,,,
2024-01-01,4174,159,North,NaN,NaN
2024-01-02,4512,468,East,NaN,NaN
2024-01-03,1870,101,North,NaN,NaN
2024-01-04,2309,484,North,NaN,NaN
2024-01-05,2150,403,South,NaN,NaN
...,...,...,...,...,...
2024-12-26,3062,124,East,4697.857143,4668.600000
2024-12-27,4592,478,North,4598.142857,4673.366667
2024-12-28,5001,152,North,4634.857143,4729.433333


In [20]:
# first n-1 rows are NaN because the window isn't full yet
# rolling(7) needs 7 rows before it can compute — rows 0-5 are NaN
df["revenue_7d"].isna().sum()  # 6 NaN values

np.int64(6)

In [21]:
# rolling works with other aggregations too
df["revenue"].rolling(7).max()   # rolling max
df["revenue"].rolling(7).std()   # rolling volatility

date
2024-01-01            NaN
2024-01-02            NaN
2024-01-03            NaN
2024-01-04            NaN
2024-01-05            NaN
                 ...     
2024-12-26     960.900867
2024-12-27     924.748151
2024-12-28     936.530731
2024-12-29     954.439402
2024-12-30    1055.222522
Name: revenue, Length: 365, dtype: float64

## 7. Shifting — Lag and Lead Comparisons


`shift(n)` moves the entire column up or down by n positions.
The data doesn't change — it just gets repositioned so you can compare each row against a past or future row **on the same row**.

---

### How it works

```
original:        shift(1):       shift(-1):
day  revenue     prev_rev        next_rev
0    100         NaN             200        ← tomorrow's value
1    200         100             150
2    150         200             300
3    300         150             250
4    250         300             NaN        ← no tomorrow
```

- `shift(1)`  → lag  — brings the past into the current row
- `shift(-1)` → lead — brings the future into the current row
- NaN appears where there's no data to shift in

---

### Why you need it

Without `shift()`, you can't compare a value to its previous row directly.
`shift()` aligns past/future values onto the same row so you can do math between them.

```python
# day-over-day change — only possible because shift aligns yesterday onto today's row
df["daily_change"] = df["revenue"] - df["revenue"].shift(1)

# % change — same idea
df["pct_change"] = df["revenue"].pct_change()  # shortcut for shift(1) division
```

---

### Common shift intervals

```python
df["revenue"].shift(1)    # yesterday (lag 1 day)
df["revenue"].shift(7)    # same day last week
df["revenue"].shift(30)   # approx same day last month
df["revenue"].shift(-1)   # tomorrow (lead 1 day)
```

---

### Lag vs Lead — when to use which

| | Use |
|--|-----|
| `shift(1)` lag | comparing today vs yesterday, growth rate, momentum |
| `shift(-1)` lead | predicting next period, checking if today predicts tomorrow |

---

### `shift()` vs `diff()`

```python
df["revenue"].shift(1)             # gives you the previous value
df["revenue"].diff(1)              # gives you current - previous (the difference)

# these are equivalent:
df["revenue"] - df["revenue"].shift(1)
df["revenue"].diff(1)
```

`diff()` is just a shortcut for the most common shift pattern.


In [22]:
# day-over-day revenue change
df["prev_day_revenue"] = df["revenue"].shift(1)       # yesterday's revenue
df["daily_change"]     = df["revenue"] - df["revenue"].shift(1)
df["pct_change"]       = df["revenue"].pct_change() * 100  # % change

df[["revenue", "prev_day_revenue", "daily_change", "pct_change"]].head(10)

,revenue,prev_day_revenue,daily_change,pct_change
date,,,,
2024-01-01,4174,NaN,NaN,NaN
2024-01-02,4512,4174.0,338.0,8.097748
2024-01-03,1870,4512.0,-2642.0,-58.554965
2024-01-04,2309,1870.0,439.0,23.475936
2024-01-05,2150,2309.0,-159.0,-6.886098
2024-01-06,2120,2150.0,-30.0,-1.395349
2024-01-07,4802,2120.0,2682.0,126.509434
2024-01-08,4127,4802.0,-675.0,-14.056643
2024-01-09,2678,4127.0,-1449.0,-35.110250


In [23]:
# week-over-week comparison (shift by 7)
df["wow_change"] = df["revenue"] - df["revenue"].shift(7)
df[["revenue", "wow_change"]].head(15)

,revenue,wow_change
date,,
2024-01-01,4174,NaN
2024-01-02,4512,NaN
2024-01-03,1870,NaN
2024-01-04,2309,NaN
2024-01-05,2150,NaN
2024-01-06,2120,NaN
2024-01-07,4802,NaN
2024-01-08,4127,-47.0
2024-01-09,2678,-1834.0


In [24]:
# shift(-1) = look ahead (lead) — tomorrow's value on today's row
df["next_day_revenue"] = df["revenue"].shift(-1)
df[["revenue", "next_day_revenue"]].tail(5)  # last row will be NaN

,revenue,next_day_revenue
date,,
2024-12-26,3062,4592.0
2024-12-27,4592,5001.0
2024-12-28,5001,5208.0
2024-12-29,5208,3443.0
2024-12-30,3443,NaN


## 8. Putting It Together — Mini Analysis

In [25]:
# Monthly summary: total revenue, avg daily users, best single day
monthly = df.resample("ME").agg(
    total_revenue = ("revenue", "sum"),
    avg_users     = ("users",   "mean"),
    best_day      = ("revenue", "max")
).reset_index()

monthly["month"] = monthly["date"].dt.month_name()
monthly[["month", "total_revenue", "avg_users", "best_day"]]

,month,total_revenue,avg_users,best_day
0,January,97111,269.032258,4825
1,February,96610,335.862069,4963
2,March,109751,288.741935,5388
3,April,114187,302.166667,5474
4,May,108033,317.387097,5618
5,June,121180,297.400000,5745
6,July,121117,283.000000,5742
7,August,136455,312.322581,6029
8,September,130198,313.766667,6218
9,October,141591,290.193548,6268


In [26]:
# which month had highest revenue?
monthly.loc[monthly["total_revenue"].idxmax(), "month"]

'October'

In [27]:
# month-over-month revenue growth %
monthly["month_growth"] = monthly["total_revenue"].pct_change() * 100
monthly[["month", "total_revenue", "month_growth"]]

,month,total_revenue,month_growth
0,January,97111,NaN
1,February,96610,-0.515904
2,March,109751,13.602112
3,April,114187,4.041877
4,May,108033,-5.389405
5,June,121180,12.169430
6,July,121117,-0.051989
7,August,136455,12.663788
8,September,130198,-4.585394
9,October,141591,8.750518


```python
pct_change() has no concept of time itself. It just computes row over row percentage change. What that means depends entirely on what your data is.


# monthly data — pct_change() = month over month
monthly["total_revenue"].pct_change()
# Jan→Feb change, Feb→Mar change...

# daily data — pct_change() = day over day
daily["revenue"].pct_change()
# Mon→Tue change, Tue→Wed change...

# yearly data — pct_change() = year over year
yearly["revenue"].pct_change()
# 2022→2023 change, 2023→2024 change...
pct_change() just does (current - previous) / previous — it doesn't look at the dates at all.

## Practice Exercises

1. Find the week with the highest total revenue using `resample()`
2. Add a column `revenue_14d` — 14-day rolling average
3. Find which quarter had the most users (sum)
4. Add a column `is_above_30d_avg` — True if that day's revenue is above the 30-day rolling average
5. Find the day with the biggest single-day revenue drop (most negative `daily_change`)

In [ ]:
# Exercise 1: Find the week with the highest total revenue using resample()
# option 1 — named agg + loc
weekly = df.resample("W").agg(
    total_revenue = ("revenue", "sum")
).reset_index()

week_with_highest_revenue = weekly.loc[weekly["total_revenue"].idxmax(), "date"]
print(week_with_highest_revenue)

In [ ]:
# Exercise 1: Find the week with the highest total revenue using resample()
# option 2 — direct sum + idxmax
weekly = df["revenue"].resample("W").sum()
print(weekly.idxmax())

In [ ]:
# Exercise 1: Find the week with the highest total revenue using resample()
# option 3 — sort_values + iloc
weekly = df["revenue"].resample("W").sum().reset_index()
weekly = weekly.sort_values("revenue", ascending=False)
print(weekly.iloc[0])

In [ ]:
# Exercise 2: Add a column 'revenue_14d' — 14-day rolling average
df["revenue_14d"] = df["revenue"].rolling(14).mean()
print(df[["revenue", "revenue_14d"]])

In [ ]:
# Exercise 3: Find which quarter had the most users (sum)
quarter = df["users"].resample("QE").sum()
quarter.idxmax()

In [ ]:
# Exercise 4: Add a column 'is_above_30d_avg' — True if that day's revenue is above the 30-day rolling average
df["revenue_30d_avg"] = df["revenue"].rolling(30).mean()
df["above_30d_avg"] = df["revenue"] > df["revenue_30d_avg"]
print(df[["revenue", "revenue_30d_avg", "above_30d_avg"]])

In [ ]:
# Exercise 5: Find the day with the biggest single-day revenue drop (most negative daily_change)
df["daily_change"] = df["revenue"] - df["revenue"].shift(1)
df["daily_change"].idxmin()